In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/multi-class/amazon/train.csv')
df.head()

,productId,Title,userId,Helpfulness,Score,Time,Text,Cat1,Cat2,Cat3
0,B000E46LYG,Golden Valley Natural Buffalo Jerky,A3MQDNGHDJU4MK,0/0,3.0,-1,The description and photo on this product need...,grocery gourmet food,meat poultry,jerky
1,B000GRA6N8,Westing Game,unknown,0/0,5.0,860630400,This was a great book!!!! It is well thought t...,toys games,games,unknown
2,B000GRA6N8,Westing Game,unknown,0/0,5.0,883008000,"I am a first year teacher, teaching 5th grade....",toys games,games,unknown
3,B000GRA6N8,Westing Game,unknown,0/0,5.0,897696000,I got the book at my bookfair at school lookin...,toys games,games,unknown
4,B00000DMDQ,I SPY A is For Jigsaw Puzzle 63pc,unknown,2/4,5.0,911865600,Hi! I'm Martine Redman and I created this puzz...,toys games,puzzles,jigsaw puzzles


In [3]:
def make_funcat_label(row):
  # Pega as categorias, ignora Nones/NaNs e converte para string
  cats = [str(row[c]) for c in ['Cat1', 'Cat2', 'Cat3'] if pd.notna(row[c]) and row[c] != 'None']
  # Junta com ponto (padrão FunCat) ou barra (dependendo da sua preferência)
  return ".".join(cats).replace(' ', '_')

In [5]:
df['hierarchy_label'] = df.apply(make_funcat_label, axis=1)

print("--- Exemplo de Rótulos FunCat Gerados ---")
df[['Text', 'hierarchy_label']].head()

--- Exemplo de Rótulos FunCat Gerados ---


,Text,hierarchy_label
0,The description and photo on this product need...,grocery_gourmet_food.meat_poultry.jerky
1,This was a great book!!!! It is well thought t...,toys_games.games.unknown
2,"I am a first year teacher, teaching 5th grade....",toys_games.games.unknown
3,I got the book at my bookfair at school lookin...,toys_games.games.unknown
4,Hi! I'm Martine Redman and I created this puzz...,toys_games.puzzles.jigsaw_puzzles


In [19]:
from sentence_transformers import SentenceTransformer

# Carrega o modelo pré-treinado
# Este modelo gera vetores de tamanho 384 (mais leve). 
# Para 768 dimensões, use 'bert-base-nli-mean-tokens'
print("Carregando modelo BERT...")
model = SentenceTransformer('bert-base-nli-mean-tokens') 

# Concatenar Título + Texto melhora a qualidade do embedding
df['full_text'] = df['Title'] + ". " + df['Text']

print("Gerando embeddings (pode demorar dependendo do tamanho do dataset)...")
embeddings = model.encode(df['full_text'].tolist(), show_progress_bar=True)

# O resultado é uma matriz numpy (N_samples, Embedding_Dim)
print(f"Shape dos Embeddings: {embeddings.shape}")

Carregando modelo BERT...


c:\Users\medei\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning:

`huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\medei\.cache\huggingface\hub\models--sentence-transformers--bert-base-nli-mean-tokens. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 940.36it/s, Materializing param=p

Gerando embeddings (pode demorar dependendo do tamanho do dataset)...


Batches:   8%|▊         | 95/1250 [02:54<35:19,  1.84s/it]


KeyboardInterrupt: 

In [8]:
embeddings

array([[-4.6015881e-02, -8.6606354e-02,  2.6284773e-03, ...,
        -4.1619826e-02,  8.1721865e-02,  1.2842039e-02],
       [ 4.5024157e-02,  1.1084073e-04,  1.1791492e-02, ...,
         1.8355902e-04, -1.1881834e-01,  6.2667713e-02],
       [ 7.6785885e-02, -1.7568352e-02,  4.3598767e-03, ...,
         8.5129432e-02, -1.2182289e-01,  7.2414123e-02],
       ...,
       [-6.6240460e-02,  3.0583331e-02,  1.0223583e-01, ...,
         7.0790090e-02, -2.9751571e-02,  5.5243861e-02],
       [-2.7775990e-02,  3.9843386e-03,  9.1720678e-02, ...,
        -5.4343119e-02,  6.6381559e-02,  3.6126915e-02],
       [-3.8513012e-02, -3.7622716e-02,  3.8680114e-02, ...,
        -1.8882522e-02,  1.0687356e-02,  2.1505214e-02]],
      shape=(40000, 384), dtype=float32)

In [9]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

In [11]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_embedded = tsne.fit_transform(embeddings)

In [13]:
df_viz = pd.DataFrame(X_embedded, columns=['x', 'y'])
df_viz['label'] = df['hierarchy_label']

df_viz.head()

,x,y,label
0,2.860131,-63.804169,grocery_gourmet_food.meat_poultry.jerky
1,110.938889,-19.136738,toys_games.games.unknown
2,110.836365,-19.060915,toys_games.games.unknown
3,110.923340,-19.076836,toys_games.games.unknown
4,76.543991,-18.590542,toys_games.puzzles.jigsaw_puzzles


In [17]:
def get_broad_category(label):
  parts = label.split('.') # ou '/' dependendo do seu formato
  # if len(parts) > 1:
  #   return parts[1] # Pega o nível logo após a Raiz
  return parts[0]

df_viz['category_group'] = df_viz['label'].apply(get_broad_category)
df_viz.head()

,x,y,label,category_group
0,2.860131,-63.804169,grocery_gourmet_food.meat_poultry.jerky,grocery_gourmet_food
1,110.938889,-19.136738,toys_games.games.unknown,toys_games
2,110.836365,-19.060915,toys_games.games.unknown,toys_games
3,110.923340,-19.076836,toys_games.games.unknown,toys_games
4,76.543991,-18.590542,toys_games.puzzles.jigsaw_puzzles,toys_games


In [18]:
fig = px.scatter(
  df_viz,
  x='x',
  y='y',
  color='category_group', # Colore pelos grandes grupos (Racism, Sexism, etc)
  hover_data=['label'], # Mostra detalhes ao passar o mouse
  title='Visualização t-SNE dos Embeddings BERT (Agrupados por Categoria Pai)',
  opacity=0.7,
  width=1000,
  height=700
)

fig.show()

In [9]:
import pandas as pd
import plotly.express as px

# 1. Simulação dos seus dados (Substitua pela sua coluna real: df['categorias'])


def analyze_dot_hierarchy(class_list):
    """
    Analisa strings no formato 'A.B.C' e gera estatísticas e dados para plotagem.
    """
    # Listas para montar o DataFrame do Plotly
    ids = []      # ID único (caminho completo: 'grocery.meat')
    labels = []   # Nome visível ('meat')
    parents = []  # ID do pai ('grocery')
    
    unique_entries = set() # Para evitar duplicatas
    level_counts = {}      # Contagem de nós por nível
    
    # Adicionar uma Raiz Abstrata para unir tudo
    root_label = "Total_Dataset"
    ids.append(root_label)
    labels.append(root_label)
    parents.append("") # Raiz não tem pai
    unique_entries.add(root_label)
    
    print(f"--- Processando {len(class_list)} exemplos ---")
    
    for row in class_list:
        if not isinstance(row, str): continue
        
        # Divide 'grocery.meat.jerky' -> ['grocery', 'meat', 'jerky']
        parts = row.split('.')
        
        current_path = ""
        parent_path = root_label
        
        for i, part in enumerate(parts):
            # O nível é i+1 (0 virá Nível 1)
            depth = i + 1
            
            # Reconstrói o caminho acumulado para usar como ID Único
            # Isso evita que 'Accessories' em 'Auto' se misture com 'Accessories' em 'Fashion'
            if i == 0:
                current_path = part
            else:
                current_path = f"{parent_path}.{part}"
            
            # Se este nó ainda não foi processado, adiciona
            if current_path not in unique_entries:
                ids.append(current_path)
                labels.append(part)      # O que vai aparecer escrito no gráfico
                parents.append(parent_path)
                unique_entries.add(current_path)
                
                # Contagem estatística
                level_counts[depth] = level_counts.get(depth, 0) + 1
            
            # Atualiza pai para a próxima iteração
            parent_path = current_path

    # Printar Estatísticas
    print("\n=== ESTATÍSTICAS DA HIERARQUIA ===")
    total_nodes = len(ids) - 1 # Remove a raiz artificial
    print(f"Total de nós únicos na árvore: {total_nodes}")
    print(f"Profundidade Máxima: {max(level_counts.keys())}")
    print("\nNós por Nível:")
    for level in sorted(level_counts.keys()):
        print(f"  - Nível {level}: {level_counts[level]} categorias")
        
    return pd.DataFrame({'ids': ids, 'labels': labels, 'parents': parents})

# Executar a análise
df_viz = analyze_dot_hierarchy(df['hierarchy_label'])

--- Processando 240942 exemplos ---

=== ESTATÍSTICAS DA HIERARQUIA ===
Total de nós únicos na árvore: 298
Profundidade Máxima: 3

Nós por Nível:
  - Nível 1: 9 categorias
  - Nível 2: 70 categorias
  - Nível 3: 219 categorias


In [16]:
def plot_treemap(df_viz):
    fig = px.treemap(
        df_viz,
        ids='ids',
        names='labels',
        parents='parents',
        title="Mapa de Categorias (Treemap)",
        width=1000,
        height=600
    )
    # Adiciona cores baseadas no primeiro nível (pais)
    fig.update_traces(root_color="lightgrey")
    fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
    
    fig.show()

plot_treemap(df_viz)

In [17]:
def plot_sunburst(df_viz):
    print("\nGerando gráfico interativo...")
    fig = px.sunburst(
        df_viz,
        ids='ids', 
        names='labels',      # Nome curto (ex: 'jerky')
        parents='parents',   # Quem é o pai
        title="Hierarquia de Categorias de Produtos",
        width=900,
        height=900
    )
    
    # Melhora o layout para leitura
    fig.update_layout(margin=dict(t=40, l=0, r=0, b=0))
    fig.update_traces(textinfo="label+percent entry") # Mostra nome e % do nível atual
    
    fig.show()

plot_sunburst(df_viz)


Gerando gráfico interativo...
